> ✅ **SOLVED VERSION**: every `# TODO` is filled in and every written answer is completed.

# 🧠 LSTM Hands-On Lab: Tasks for Students

**Goal:** after this lab you should be able to
1. explain what the gates of an LSTM do and why they fix the vanishing-gradient problem,
2. implement an LSTM cell from scratch,
3. count LSTM parameters by hand,
4. train LSTMs for time-series prediction, sequence memory, and text generation,
5. compare LSTM with a vanilla RNN / GRU and reason about the results.

**How this notebook works**
- Each section has a short **📘 Learn** part, followed by one or more **✏️ Task** cells.
- Fill in every place marked `# TODO`. Replace `...` / `None` with your own code.
- After most tasks there is a **✅ Check** cell. It should print `✔ passed`. Do not edit it.
- Answer written questions in the cells marked **Your answer:**.

| Section | Topic | Difficulty |
|---|---|---|
| 0 | Setup | – |
| 1 | Concept check | ⭐ |
| 2 | LSTM cell from scratch (NumPy) | ⭐⭐ |
| 3 | Counting parameters | ⭐⭐ |
| 4 | Sine-wave forecasting (PyTorch) | ⭐⭐ |
| 5 | Long-term memory: RNN vs LSTM | ⭐⭐⭐ |
| 6 | Character-level text generation | ⭐⭐⭐ |
| 7 | Free experiments + reflection | ⭐⭐ |

## 0. Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

## 1. Concept check

### 📘 Learn
A vanilla RNN updates one hidden state: $h_t = \tanh(W x_t + U h_{t-1} + b)$.
During backpropagation through time the gradient is multiplied by (roughly) $U^\top \cdot \text{diag}(\tanh')$ **at every step**, so over many steps it **vanishes** (or explodes).

An **LSTM** adds a separate **cell state** $c_t$ (the "memory highway") and three **gates** (values between 0 and 1):

$$
\begin{aligned}
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) &&\text{input gate: how much new info to write}\\
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) &&\text{forget gate: how much old memory to keep}\\
g_t &= \tanh(W_g x_t + U_g h_{t-1} + b_g) &&\text{candidate memory}\\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) &&\text{output gate: how much memory to expose}\\[4pt]
c_t &= f_t \odot c_{t-1} + i_t \odot g_t\\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

The key line is $c_t = f_t \odot c_{t-1} + i_t \odot g_t$: it is **additive**, so the gradient $\partial c_t / \partial c_{t-1} = f_t$ can stay close to 1 and flow through many time steps.

### ✏️ Task 1: Answer in 1–3 sentences each
**Q1.** If $f_t = 1$ and $i_t = 0$ for all $t$, what happens to the cell state $c_t$?

**Q2.** If $f_t = 0$ for all $t$, what does the LSTM behave like?

**Q3.** Why is the forget-gate bias often initialised to a positive number (e.g. 1)?

**Q4.** What is the difference between the *hidden state* $h_t$ and the *cell state* $c_t$? Which one is passed to the next layer / the output layer?

**Your answers:**

**A1.** The cell state never changes: c_t = 1·c_{t-1} + 0·g_t = c_{t-1}, so c_t = c_0 for all t. The memory is carried forward perfectly and nothing new is written (and the gradient ∂c_t/∂c_{t-1} = 1, so it does not vanish).

**A2.** With f_t = 0 the old memory is wiped at every step: c_t = i_t ⊙ g_t. The cell has no long-term memory and acts like a memoryless function of the current input (plus whatever h_{t-1} the gates see), i.e. very close to a vanilla RNN / feed-forward layer.

**A3.** With a positive bias, σ(b_f) ≈ 0.73 for b_f = 1, so at the start of training the forget gate is mostly "open" (remember by default). The gradient then flows through c_t = f_t ⊙ c_{t-1} + … over many steps instead of being cut at initialisation, which makes learning long-range dependencies much easier. The network can still learn to forget later.

**A4.** The cell state c_t is the internal long-term memory (the "highway"); it is not shown directly to other layers. The hidden state h_t = o_t ⊙ tanh(c_t) is a filtered (gated) read-out of that memory. h_t is what is passed to the next layer / the output layer (and also fed back at the next time step together with c_t).


## 2. LSTM cell from scratch (NumPy)

### 📘 Learn
In practice all four gates are computed with **one** big matrix multiplication and then split:

$$ z_t = W x_t + U h_{t-1} + b \in \mathbb{R}^{4H} $$

Then $z_t$ is cut into 4 chunks of size $H$. We use **PyTorch's order**: `i, f, g, o`.

Shapes (D = input size, H = hidden size):
- `x_t`: `(D,)`, `h_prev`, `c_prev`: `(H,)`
- `W`: `(4H, D)`, `U`: `(4H, H)`, `b`: `(4H,)`

### ✏️ Task 2a: Implement one LSTM step

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def lstm_cell_forward(x_t, h_prev, c_prev, W, U, b):
    '''One LSTM time step. Returns (h_t, c_t).'''
    H = h_prev.shape[0]

    # SOLUTION 1: compute the pre-activations z (shape (4H,))
    z = W @ x_t + U @ h_prev + b

    # SOLUTION 2: split z into 4 chunks (order: i, f, g, o) and apply the right activation to each
    i = sigmoid(z[0:H])        # input gate
    f = sigmoid(z[H:2*H])      # forget gate
    g = np.tanh(z[2*H:3*H])    # candidate memory
    o = sigmoid(z[3*H:4*H])    # output gate

    # SOLUTION 3: new cell state and hidden state
    c_t = f * c_prev + i * g
    h_t = o * np.tanh(c_t)
    return h_t, c_t

### ✅ Check 2a: compare with PyTorch's `nn.LSTMCell`

In [ ]:
D, H = 3, 4
torch_cell = nn.LSTMCell(D, H)
W = torch_cell.weight_ih.detach().numpy()                       # (4H, D)
U = torch_cell.weight_hh.detach().numpy()                       # (4H, H)
b = (torch_cell.bias_ih + torch_cell.bias_hh).detach().numpy()  # (4H,)

x  = np.random.randn(D).astype(np.float32)
h0 = np.random.randn(H).astype(np.float32)
c0 = np.random.randn(H).astype(np.float32)

h_mine, c_mine = lstm_cell_forward(x, h0, c0, W, U, b)
h_ref, c_ref = torch_cell(torch.tensor(x)[None], (torch.tensor(h0)[None], torch.tensor(c0)[None]))

assert np.allclose(h_mine, h_ref.detach().numpy()[0], atol=1e-5), "h_t is wrong"
assert np.allclose(c_mine, c_ref.detach().numpy()[0], atol=1e-5), "c_t is wrong"
print("✔ passed")

### ✏️ Task 2b: Run the cell over a whole sequence
Write `lstm_forward(X, W, U, b)` that loops over a sequence `X` of shape `(T, D)` starting from zero states and returns all hidden states `(T, H)` plus the final `c`.

In [ ]:
def lstm_forward(X, W, U, b):
    T, D = X.shape
    H = U.shape[1]
    h = np.zeros(H, dtype=np.float32)
    c = np.zeros(H, dtype=np.float32)
    all_h = []
    for t in range(T):
        h, c = lstm_cell_forward(X[t], h, c, W, U, b)
        all_h.append(h)
    return np.stack(all_h), c

In [ ]:
# ✅ Check 2b
X = np.random.randn(6, D).astype(np.float32)
H_mine, c_final = lstm_forward(X, W, U, b)

lstm = nn.LSTM(D, H, batch_first=True)
with torch.no_grad():
    lstm.weight_ih_l0.copy_(torch_cell.weight_ih); lstm.weight_hh_l0.copy_(torch_cell.weight_hh)
    lstm.bias_ih_l0.copy_(torch_cell.bias_ih);     lstm.bias_hh_l0.copy_(torch_cell.bias_hh)
    out, (hn, cn) = lstm(torch.tensor(X)[None])

assert H_mine.shape == (6, H)
assert np.allclose(H_mine, out[0].numpy(), atol=1e-5), "Hidden states differ from nn.LSTM"
assert np.allclose(c_final, cn[0, 0].numpy(), atol=1e-5), "Final cell state differs"
print("✔ passed")

### ✏️ Task 2c: Watch the gates (exploration)
Set the forget-gate bias very **high** (+10) and then very **low** (−10) and feed the same sequence. Plot $c_t$ over time for one hidden unit in both cases. *(Tip: the forget gate is the 2nd chunk of `b`, i.e. `b[H:2*H]`.)*

What do you observe? Write one sentence below the plot.

In [ ]:
def run_and_collect_c(X, W, U, b):
    '''Same as lstm_forward but returns the cell state c_t at every step: array (T, H).'''
    T = X.shape[0]
    H = U.shape[1]
    h = np.zeros(H, dtype=np.float32)
    c = np.zeros(H, dtype=np.float32)
    all_c = []
    for t in range(T):
        h, c = lstm_cell_forward(X[t], h, c, W, U, b)
        all_c.append(c)
    return np.stack(all_c)

X_long = np.random.randn(60, D).astype(np.float32)

b_high = b.copy(); b_high[H:2*H] = 10.0     # forget gate ~ 1  -> remember everything
b_low  = b.copy(); b_low[H:2*H]  = -10.0    # forget gate ~ 0  -> forget everything

c_high = run_and_collect_c(X_long, W, U, b_high)
c_low  = run_and_collect_c(X_long, W, U, b_low)

plt.figure(figsize=(10, 3.5))
plt.plot(c_high[:, 0], label="forget bias = +10 (f ≈ 1)")
plt.plot(c_low[:, 0],  label="forget bias = -10 (f ≈ 0)")
plt.xlabel("time step"); plt.ylabel("c_t (unit 0)"); plt.title("Cell state of unit 0 for two forget-gate biases")
plt.legend(); plt.show()

**Observation:** With a high forget bias (f ≈ 1) the cell state accumulates over time like a running sum, so it drifts smoothly and keeps the whole history. With a very low forget bias (f ≈ 0) the cell state is rewritten at every step (c_t ≈ i_t·g_t), so it just jumps around following the current input and has no memory.

## 3. Counting parameters

### 📘 Learn
For one LSTM layer with input size $D$ and hidden size $H$:

- 4 gates, each with an input matrix $(H \times D)$, a recurrent matrix $(H \times H)$ and a bias $(H)$.
- PyTorch uses **two** bias vectors per gate (`bias_ih` and `bias_hh`), so the total biases are $8H$.

For layers after the first, the input size is $H$ (or $2H$ if bidirectional).

### ✏️ Task 3: Write `count_lstm_params(D, H, num_layers=1, bidirectional=False)`

In [ ]:
def count_lstm_params(D, H, num_layers=1, bidirectional=False):
    dirs = 2 if bidirectional else 1
    total = 0
    for layer in range(num_layers):
        in_size = D if layer == 0 else H * dirs          # later layers see the (concatenated) outputs
        # per direction: W_ih (4H x in) + W_hh (4H x H) + b_ih (4H) + b_hh (4H)
        total += dirs * (4 * H * in_size + 4 * H * H + 8 * H)
    return total

In [ ]:
# ✅ Check 3
for (D, H, L, bi) in [(1, 8, 1, False), (10, 32, 1, False), (10, 32, 2, False), (5, 16, 3, True)]:
    m = nn.LSTM(D, H, num_layers=L, bidirectional=bi)
    real = sum(p.numel() for p in m.parameters())
    mine = count_lstm_params(D, H, L, bi)
    assert real == mine, f"D={D}, H={H}, L={L}, bi={bi}: expected {real}, got {mine}"
print("✔ passed")

**Answer: RNN = 1408, GRU = 4224 (3×), LSTM = 5632 (4×).** (RNN: 32·10 + 32·32 + 2·32 = 1408; GRU has 3 sets of these weights, LSTM has 4.)

**Thinking question:** an LSTM has ~4× the parameters of a vanilla RNN with the same sizes, and a GRU has ~3×. Verify this quickly with `nn.RNN`, `nn.GRU`, `nn.LSTM` for `D=10, H=32` and write the three numbers below.

In [ ]:
for name, cls in [("RNN", nn.RNN), ("GRU", nn.GRU), ("LSTM", nn.LSTM)]:
    m = cls(10, 32)
    print(f"{name:5s}: {sum(p.numel() for p in m.parameters()):5d} parameters")

## 4. Sine-wave forecasting with PyTorch

### 📘 Learn
Typical PyTorch LSTM recipe (`batch_first=True`):

| Tensor | Shape |
|---|---|
| input to `nn.LSTM` | `(batch, seq_len, input_size)` |
| `output` | `(batch, seq_len, hidden_size)`, the hidden state at **every** step |
| `h_n`, `c_n` | `(num_layers, batch, hidden_size)`, the **final** states |

For **many-to-one** prediction (predict the next value) we take the last time step: `output[:, -1, :]`, and pass it to a `nn.Linear`.

### ✏️ Task 4a: Build the dataset
We have a noisy sine wave. Use a sliding window: the input is `window` consecutive values, the target is the **next** value.

In [ ]:
t = np.linspace(0, 100, 2000)
series = np.sin(t) + 0.1 * np.random.randn(len(t))
series = series.astype(np.float32)

plt.figure(figsize=(10, 3)); plt.plot(series[:300]); plt.title("Noisy sine wave (first 300 points)"); plt.show()

def make_windows(series, window):
    '''Return X of shape (N, window, 1) and y of shape (N, 1).'''
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])
        y.append(series[i + window])
    X = np.array(X, dtype=np.float32)[..., None]
    y = np.array(y, dtype=np.float32)[:, None]
    return X, y

WINDOW = 30
X, y = make_windows(series, WINDOW)

# chronological split (never shuffle time series before splitting!)
split = int(0.8 * len(X))
X_train, y_train = torch.tensor(X[:split]), torch.tensor(y[:split])
X_test,  y_test  = torch.tensor(X[split:]), torch.tensor(y[split:])
print(X_train.shape, y_train.shape, X_test.shape)

In [ ]:
# ✅ Check 4a
assert X.shape == (len(series) - WINDOW, WINDOW, 1), "wrong number of windows"
assert np.isclose(X[0, 0, 0], series[0]) and np.isclose(y[0, 0], series[WINDOW]), "window/target misaligned"
print("✔ passed")

### ✏️ Task 4b: Define the model

In [ ]:
class SineLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)            # (batch, seq, hidden)
        return self.fc(out[:, -1, :])    # last time step -> (batch, 1)

In [ ]:
# ✅ Check 4b
m = SineLSTM()
assert m(torch.randn(5, WINDOW, 1)).shape == (5, 1), "output should be (batch, 1)"
print("✔ passed")

### ✏️ Task 4c: Train it
Use `MSELoss` and `Adam` (lr=0.01), mini-batches of 64, around 15 epochs. Record the train and test loss each epoch.

In [ ]:
def train_regressor(model, X_train, y_train, X_test, y_test, epochs=15, lr=1e-2, batch_size=64):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_hist, test_hist = [], []

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X_train))
        total = 0.0
        for k in range(0, len(X_train), batch_size):
            idx = perm[k:k + batch_size]
            xb, yb = X_train[idx].to(device), y_train[idx].to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total += loss.item() * len(idx)

        model.eval()
        with torch.no_grad():
            test_loss = criterion(model(X_test.to(device)), y_test.to(device)).item()
        train_hist.append(total / len(X_train)); test_hist.append(test_loss)
        print(f"epoch {epoch+1:2d} | train {train_hist[-1]:.4f} | test {test_loss:.4f}")
    return train_hist, test_hist

torch.manual_seed(SEED)
model = SineLSTM(hidden_size=32)
train_hist, test_hist = train_regressor(model, X_train, y_train, X_test, y_test)

> 💡 **Gradient clipping.** LSTMs fix *vanishing* gradients but can still have *exploding* ones. Add
> `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` between `backward()` and `step()`.

In [ ]:
# ✅ Check 4c
assert test_hist[-1] < 0.05, f"Test MSE is {test_hist[-1]:.3f}, expected < 0.05 (noise variance is 0.01). Check your loop."
print("✔ passed")

### ✏️ Task 4d: Visualise predictions and do a multi-step forecast

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X_test.to(device)).cpu().numpy()

# SOLUTION 1: predictions vs truth (first 200 test points)
plt.figure(figsize=(10, 3.5))
plt.plot(y_test[:200].numpy(), label="true")
plt.plot(pred[:200], label="predicted (1 step ahead)")
plt.legend(); plt.title("One-step-ahead prediction on the test set"); plt.show()

# SOLUTION 2: recursive forecasting
def forecast(model, last_window, steps=100):
    """last_window: tensor (WINDOW, 1). Returns a list of `steps` predicted floats."""
    model.eval()
    window = last_window.clone().to(device)
    preds = []
    with torch.no_grad():
        for _ in range(steps):
            nxt = model(window.unsqueeze(0)).item()                       # predict one step
            preds.append(nxt)
            new = torch.tensor([[nxt]], dtype=window.dtype, device=device)
            window = torch.cat([window[1:], new], dim=0)                  # drop oldest, append prediction
    return preds

# We start 100 steps before the END of the test set, so that the true continuation
# (y_test[-100:]) exists and can be compared with the forecast.
STEPS = 100
fc = forecast(model, X_test[-STEPS], steps=STEPS)

# SOLUTION 3: forecast vs the true continuation
plt.figure(figsize=(10, 3.5))
plt.plot(y_test[-STEPS:].numpy(), label="true continuation")
plt.plot(fc, label="recursive 100-step forecast")
plt.legend(); plt.title("Recursive multi-step forecast"); plt.show()
print("MSE of the 100-step forecast:", float(np.mean((np.array(fc) - y_test[-STEPS:].numpy().ravel()) ** 2)))

**Question:** does the recursive forecast stay accurate for all 100 steps? Why do errors usually accumulate?

**Your answer:** For the first part of the horizon the forecast usually follows the sine well, because the pattern is simple and periodic. Over many steps it typically drifts (small phase shift or amplitude damping) because every prediction is fed back as input: each small error becomes part of the next input, so errors compound and the model is running on inputs it never saw in training (it was trained on *noisy true* values, not on its own smooth predictions). The longer the horizon, the larger the accumulated error.

## 5. Long-term memory: RNN vs LSTM

### 📘 Learn
We build a task where the answer depends on the **very first** element of a long sequence:

- Input: a sequence of `T` random digits (0–9), one-hot encoded.
- Target: the **first** digit.

To solve this the network must carry information across `T-1` steps of useless noise. A vanilla RNN struggles as `T` grows; the LSTM should keep working.

In [ ]:
def make_memory_data(n, T, n_classes=10):
    digits = torch.randint(0, n_classes, (n, T))
    X = torch.nn.functional.one_hot(digits, n_classes).float()   # (n, T, 10)
    y = digits[:, 0]                                             # (n,)
    return X, y

Xm, ym = make_memory_data(4, 6)
print(Xm.shape, ym)

### ✏️ Task 5a: A model that can be either an RNN or an LSTM
Write `MemoryNet(rnn_type, ...)` where `rnn_type ∈ {"rnn", "lstm", "gru"}`. It reads the sequence and classifies using the final time step.

In [ ]:
class MemoryNet(nn.Module):
    def __init__(self, rnn_type="lstm", input_size=10, hidden_size=64, n_classes=10):
        super().__init__()
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

In [ ]:
# ✅ Check 5a
for kind in ["rnn", "lstm", "gru"]:
    assert MemoryNet(kind)(torch.randn(3, 7, 10)).shape == (3, 10), f"{kind}: wrong output shape"
print("✔ passed")

### ✏️ Task 5b: Train and compare
Complete `train_memory` (cross-entropy, Adam lr=3e-3, gradient clipping 1.0). Fresh random data is generated every step, so there is no overfitting.

In [ ]:
def train_memory(rnn_type, T, steps=600, batch_size=64, lr=3e-3, hidden=64):
    torch.manual_seed(SEED)
    model = MemoryNet(rnn_type, hidden_size=hidden).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for step in range(steps):
        Xb, yb = make_memory_data(batch_size, T)
        Xb, yb = Xb.to(device), yb.to(device)
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # final accuracy on fresh data
    model.eval()
    with torch.no_grad():
        Xt, yt = make_memory_data(2000, T)
        acc = (model(Xt.to(device)).argmax(1).cpu() == yt).float().mean().item()
    return acc

# chance level is 0.10
results = {}
for kind in ["rnn", "lstm", "gru"]:
    for T in [5, 20, 50]:
        results[(kind, T)] = train_memory(kind, T)
        print(f"{kind:>4} | T={T:3d} | accuracy = {results[(kind, T)]:.2f}")

In [ ]:
Ts = [5, 20, 50]
plt.figure(figsize=(6, 4))
for kind in ["rnn", "lstm", "gru"]:
    plt.plot(Ts, [results[(kind, T)] for T in Ts], marker="o", label=kind)
plt.axhline(0.10, color="gray", ls="--", label="chance")
plt.xlabel("sequence length T"); plt.ylabel("accuracy"); plt.ylim(0, 1.05)
plt.title("Remember the first digit: accuracy vs sequence length"); plt.legend(); plt.show()

**Questions**
1. At which `T` does the vanilla RNN start to fail? Does the LSTM fail too if you push `T` to 100 or 200? (try it, you may need more `steps`)
2. Which of the 3 models trains fastest for small `T`? Why might that be?

**Your answers:** *(The exact numbers depend on your run: compare with the table printed above. Typical behaviour:)*

1. The vanilla RNN is fine for very short sequences (T = 5), starts to degrade around T ≈ 20 and is close to chance (0.10) at T = 50, because the gradient from the target back to the first input vanishes. The LSTM (and GRU) keep working much longer; if you push T to 100–200 they eventually fail as well unless you train longer (more `steps`), use a larger hidden size or a better initialisation (e.g. forget-gate bias = 1), because the signal still has to survive many steps.
2. For small T the plain RNN (and often the GRU) trains fastest: fewer parameters (RNN 1×, GRU 3×, LSTM 4×) and fewer gates to learn, and at short range there is no vanishing-gradient problem to solve. The extra gating of an LSTM only pays off when the dependency is long.

### ✏️ Task 5c (challenge): Measure the vanishing gradient directly
Take a trained-or-untrained `MemoryNet` and a batch with `T = 50`. Make the input require gradients, compute the loss, call `backward()`, and compute for each time step $t$ the norm $\|\partial L / \partial x_t\|$. Plot it on a **log** y-axis for `rnn` and `lstm`.

You should see that for the RNN the gradient at early time steps is much smaller than at late ones.

In [ ]:
def grad_norm_per_timestep(rnn_type, T=50, batch_size=64):
    torch.manual_seed(SEED)
    model = MemoryNet(rnn_type).to(device)
    Xb, yb = make_memory_data(batch_size, T)
    Xb = Xb.to(device).requires_grad_(True)
    yb = yb.to(device)
    model.train()                                   # (cuDNN RNN backward needs train mode)
    logits = model(Xb)
    loss = nn.functional.cross_entropy(logits, yb)
    loss.backward()
    # gradient norm for each time step: sqrt(sum over batch and features of grad^2) -> shape (T,)
    return torch.sqrt((Xb.grad ** 2).sum(dim=(0, 2))).cpu().numpy()

plt.figure(figsize=(8, 4))
for kind in ["rnn", "lstm"]:
    plt.semilogy(grad_norm_per_timestep(kind), label=kind)
plt.xlabel("time step t (0 = first / oldest input)"); plt.ylabel(r"$\|\partial L / \partial x_t\|$ (log scale)")
plt.title("Gradient reaching each input time step (T = 50)"); plt.legend(); plt.show()

**What do you see?** For the vanilla RNN the gradient norm shrinks by orders of magnitude as we go back in time: the earliest steps receive an almost zero gradient, so the network cannot learn from them (vanishing gradient). For the LSTM the curve is much flatter, because the additive cell-state path (c_t = f·c_{t-1} + i·g) lets the gradient flow back without being multiplied by the same weight matrix at every step.

## 6. Character-level text generation

### 📘 Learn
A **language model** predicts the next character from the previous ones. Once trained we can **generate** text: feed a seed, sample the next character from the output distribution, append it, repeat.

**Temperature** $\tau$ controls randomness: sample from $\text{softmax}(\text{logits}/\tau)$.
- $\tau \to 0$: always the most likely character (repetitive)
- $\tau = 1$: the model's true distribution
- $\tau > 1$: more random / creative (and more errors)

Here we use a tiny text. For better results later, replace it with a larger text file (e.g. a book from Project Gutenberg, or your own writing).

In [ ]:
text = (
    "the quick brown fox jumps over the lazy dog. "
    "a long short term memory network keeps information for a long time. "
    "the forget gate decides what to throw away and the input gate decides what to store. "
    "the cell state carries information through time and the output gate decides what to show. "
    "recurrent networks read one token at a time and remember the past. "
) * 20
print(len(text), "characters")

### ✏️ Task 6a: Vocabulary and dataset

In [ ]:
# SOLUTION 1: vocabulary
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
vocab_size = len(chars)

SEQ_LEN = 40

# SOLUTION 2: inputs and next-character targets (shifted by one)
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
starts = range(0, len(data) - SEQ_LEN - 1)
X_txt = torch.stack([data[i:i + SEQ_LEN] for i in starts])
Y_txt = torch.stack([data[i + 1:i + SEQ_LEN + 1] for i in starts])
print(vocab_size, X_txt.shape, Y_txt.shape)

In [ ]:
# ✅ Check 6a
assert all(itos[stoi[c]] == c for c in chars)
assert X_txt.shape == Y_txt.shape and X_txt.shape[1] == SEQ_LEN
assert (X_txt[0, 1:] == Y_txt[0, :-1]).all(), "targets must be inputs shifted by one"
print("✔ passed")

### ✏️ Task 6b: Model
Architecture: `Embedding → LSTM (2 layers, dropout 0.2) → Linear(vocab)`.
This time the LSTM outputs are used at **every** time step (many-to-many), so the logits should have shape `(batch, seq_len, vocab)`.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, emb=32, hidden=128, layers=2, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb)
        self.lstm = nn.LSTM(emb, hidden, num_layers=layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x, state=None):
        e = self.emb(x)                       # (batch, seq, emb)
        out, state = self.lstm(e, state)      # (batch, seq, hidden)
        return self.fc(out), state            # logits (batch, seq, vocab) + new state

In [ ]:
# ✅ Check 6b
m = CharLSTM(vocab_size)
logits, st = m(torch.randint(0, vocab_size, (4, SEQ_LEN)))
assert logits.shape == (4, SEQ_LEN, vocab_size)
print("✔ passed")

### ✏️ Task 6c: Train
Flatten logits to `(batch*seq, vocab)` and targets to `(batch*seq,)` for `CrossEntropyLoss`. Aim for a loss below ~0.5 on this tiny text.

In [ ]:
def train_char_model(model, X, Y, epochs=8, lr=3e-3, batch_size=64):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        perm = torch.randperm(len(X)); total = 0.0
        for k in range(0, len(X), batch_size):
            idx = perm[k:k + batch_size]
            xb, yb = X[idx].to(device), Y[idx].to(device)
            model.train()
            optimizer.zero_grad()
            logits, _ = model(xb)
            loss = criterion(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item() * len(idx)
        print(f"epoch {epoch+1} | loss {total/len(X):.4f}")

torch.manual_seed(SEED)
char_model = CharLSTM(vocab_size)
train_char_model(char_model, X_txt, Y_txt)

### ✏️ Task 6d: Generate text with temperature

In [ ]:
@torch.no_grad()
def generate(model, seed, length=200, temperature=1.0):
    model.eval()
    out = list(seed)
    # SOLUTION 1: warm up the state with the whole seed
    x = torch.tensor([[stoi[c] for c in seed]], device=device)
    logits, state = model(x)
    # SOLUTION 2: sample one character at a time, carrying the state
    for _ in range(length):
        probs = torch.softmax(logits[0, -1] / temperature, dim=-1)
        idx = torch.multinomial(probs, num_samples=1).item()
        out.append(itos[idx])
        x = torch.tensor([[idx]], device=device)          # feed ONLY the new character
        logits, state = model(x, state)
    return "".join(out)

for tau in [0.3, 0.8, 1.5]:
    print(f"--- temperature {tau} ---")
    print(generate(char_model, "the forget gate ", 200, tau), "\n")

**Questions**
1. How does the text change with temperature?
2. Why do we pass `state` between calls instead of re-feeding the full history each time? Is the result identical?

**Your answers:** 1. Low temperature (0.3) gives the most likely character every time: very correct but repetitive, it tends to loop over the same sentence from the training text. Temperature 0.8 is a good balance (mostly valid words, some variation). High temperature (1.5) flattens the distribution, so the text becomes more random, with spelling mistakes and non-words.
2. The LSTM state (h, c) is a summary of the whole history, so passing it on costs O(1) per generated character, whereas re-feeding the full history costs O(n) each time (O(n²) in total). In eval mode (no dropout) the result is mathematically identical, up to tiny floating-point differences, as long as the state is carried from the very beginning of the text.

## 7. Free experiments & reflection

### ✏️ Task 7a: Ablation on the sine task (Section 4)
Pick **at least three** of the experiments below, run them, and fill in the table with the final test MSE.

| Experiment | Change | Final test MSE | Notes |
|---|---|---|---|
| Baseline | `hidden=32`, 1 layer | | |
| Bigger hidden | `hidden=128` | | |
| Deeper | `num_layers=2` | | |
| Shorter window | `WINDOW=5` | | |
| Longer window | `WINDOW=100` | | |
| Replace `nn.LSTM` by `nn.GRU` | | | |
| Replace `nn.LSTM` by `nn.RNN` | | | |
| No gradient clipping | | | |

In [ ]:
import pandas as pd

class SeqRegressor(nn.Module):
    """Generic many-to-one regressor so that we can swap LSTM / GRU / RNN."""
    def __init__(self, rnn_type="lstm", hidden_size=32, num_layers=1, bidirectional=False):
        super().__init__()
        cls = {"lstm": nn.LSTM, "gru": nn.GRU, "rnn": nn.RNN}[rnn_type]
        self.rnn = cls(1, hidden_size, num_layers, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_size * (2 if bidirectional else 1), 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

def run_experiment(window=30, hidden=32, layers=1, rnn_type="lstm", clip=True,
                   bidirectional=False, epochs=15, lr=1e-2, batch_size=64):
    torch.manual_seed(SEED)
    Xw, yw = make_windows(series, window)
    cut = int(0.8 * len(Xw))
    Xtr, ytr = torch.tensor(Xw[:cut]).to(device), torch.tensor(yw[:cut]).to(device)
    Xte, yte = torch.tensor(Xw[cut:]).to(device), torch.tensor(yw[cut:]).to(device)

    net = SeqRegressor(rnn_type, hidden, layers, bidirectional).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(epochs):
        net.train()
        perm = torch.randperm(len(Xtr), device=device)
        for k in range(0, len(Xtr), batch_size):
            idx = perm[k:k + batch_size]
            optimizer.zero_grad()
            loss = criterion(net(Xtr[idx]), ytr[idx])
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
    net.eval()
    with torch.no_grad():
        return criterion(net(Xte), yte).item()

experiments = {
    "Baseline (hidden=32, 1 layer)": dict(),
    "Bigger hidden (128)":           dict(hidden=128),
    "Deeper (2 layers)":             dict(layers=2),
    "Shorter window (5)":            dict(window=5),
    "Longer window (100)":           dict(window=100),
    "GRU instead of LSTM":           dict(rnn_type="gru"),
    "Vanilla RNN instead of LSTM":   dict(rnn_type="rnn"),
    "No gradient clipping":          dict(clip=False),
}
rows = []
for name, kwargs in experiments.items():
    mse = run_experiment(**kwargs)
    rows.append((name, round(mse, 4)))
    print(f"{name:32s} test MSE = {mse:.4f}")
pd.DataFrame(rows, columns=["Experiment", "Final test MSE"])

# ---- Task 7b: effect of bidirectional=True
for bi in [False, True]:
    m = nn.LSTM(1, 32, batch_first=True, bidirectional=bi)
    o, _ = m(torch.randn(2, 30, 1))
    print(f"bidirectional={bi!s:5} | output shape {tuple(o.shape)} | parameters {sum(p.numel() for p in m.parameters())}")

### ✏️ Task 7b: Bidirectional LSTM
Set `bidirectional=True` in one of your models. What happens to (i) the output feature size of the LSTM, (ii) the number of parameters? Does a bidirectional LSTM make sense for *forecasting the future*? For *classifying a sentence*? Explain.

**Your answer:** (i) The LSTM output size doubles from H to 2H (the forward and backward hidden states are concatenated; with H = 32 the output is 64-dim, so the next Linear layer needs 2H inputs). (ii) The number of parameters roughly doubles (one full set of weights per direction: for D = 1, H = 32 it goes from 4,480 to 8,960). For *forecasting the future* a bidirectional LSTM only makes sense if it reads a window that lies completely in the past (no leakage); it is wrong for any setup where the backward direction would see values that are in the future of the point being predicted (e.g. step-by-step / streaming prediction, language modelling). For *classifying a sentence* it is a very good choice: the whole sentence is available at once and the meaning of a word depends on both left and right context.

### ✏️ Task 7c: Final reflection (4–6 sentences)
In your own words: **why does an LSTM remember better than a vanilla RNN?** Mention the cell state, the forget gate and the gradient flow. Then name one task where you would prefer a GRU or a Transformer over an LSTM and say why.

**Your answer:** An LSTM remembers better because it has a separate cell state c_t that is updated additively, c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t, instead of being completely rewritten (and squashed by tanh) at every step like the hidden state of a vanilla RNN. The forget gate decides, per memory slot, how much of the old content is kept, so the network can keep a value for hundreds of steps by learning f ≈ 1. During backpropagation the gradient along the cell-state path is multiplied by f_t (not by a weight matrix and a tanh derivative at every step), so it can stay close to 1 and does not vanish as quickly. I would prefer a GRU when I need a smaller/faster model with similar quality (fewer parameters, e.g. on an edge device), and a Transformer for long documents or large-scale NLP, where attention can look at any position directly, training is parallel over time and pre-trained models are available.

---
🎉 **Done!** Before submitting: *Kernel → Restart & Run All* and make sure every ✅ Check prints `✔ passed`.